In [1]:
import pickle
import numpy as np
import time

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.metrics import (
    f1_score, balanced_accuracy_score, roc_auc_score
)
from imblearn.over_sampling import SMOTE

In [2]:
with open('/kaggle/input/notebooks/matildeteixeira/wesad-preprocess/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

df           = data['df']
feature_cols = data['feature_cols']

# dados COMPLETOS (100%, sem subsample)
X_full      = df[feature_cols].values
y_full      = df['label'].values
groups_full = df['subject'].values

folds_full = list(LeaveOneGroupOut().split(X_full, y_full, groups_full))
print(f"Amostras: {len(y_full)} | Folds: {len(folds_full)}")

Amostras: 27543 | Folds: 15


In [3]:
# melhor combinação já encontrada no teu tuning a 40%
best_params = {'C': 0.1, 'kernel': 'poly', 'gamma': 'scale',
               'degree': 3, 'class_weight': 'balanced'}

def eval_fixed(params, use_smote):
    res = []
    for tr, te in folds_full:
        Xtr, Xte = X_full[tr], X_full[te]
        ytr, yte = y_full[tr], y_full[te]

        sc, im = StandardScaler(), SimpleImputer(strategy='median')
        Xtr = im.fit_transform(sc.fit_transform(Xtr))
        Xte = im.transform(sc.transform(Xte))

        if use_smote:
            Xtr, ytr = SMOTE(random_state=42).fit_resample(Xtr, ytr)

        m = SVC(**params, cache_size=1000, probability=True)
        m.fit(Xtr, ytr)
        yp = m.predict(Xte)
        pr = m.predict_proba(Xte)[:, 1]

        res.append({
            'f1':      f1_score(yte, yp, average='weighted', zero_division=0),
            'bal_acc': balanced_accuracy_score(yte, yp),
            'roc_auc': roc_auc_score(yte, pr),
        })

    f1  = np.mean([r['f1']      for r in res])
    ba  = np.mean([r['bal_acc'] for r in res])
    auc = np.mean([r['roc_auc'] for r in res])
    f1s = np.std([r['f1']       for r in res])
    print(f"{'SMOTE   ' if use_smote else 'NO SMOTE'} | "
          f"F1={f1:.3f} (±{f1s:.3f})  BalAcc={ba:.3f}  AUC={auc:.3f}")

for s in [True, False]:
    t = time.time()
    eval_fixed(best_params, s)
    print(f"  tempo: {(time.time()-t)/60:.1f} min\n")

SMOTE    | F1=0.784 (±0.146)  BalAcc=0.777  AUC=0.914
  tempo: 73.5 min

NO SMOTE | F1=0.790 (±0.142)  BalAcc=0.779  AUC=0.921
  tempo: 47.9 min

